In [1]:
import pandas as pd
from typing import List, Dict, Any, Union
#from statsbombpy import sb
import numpy as np
import json
import re
#N(reaS^!.sTijg7

In [3]:
def extract_matches_to_dataframe(match_data_list: List[List]) -> pd.DataFrame:
    """
    Extract match information from nested lists and convert to a pandas DataFrame.
    
    Expected structure for each match:
    [competition_info, datetime, season, teams_dict, home_events_list, away_events_list]
    
    Args:
        match_data_list: List of lists, where each inner list represents a match
    
    Returns:
        pandas.DataFrame with all match and event information
    """
    all_rows = []
    
    for match_idx, match in enumerate(match_data_list):
        try:
            # Extract basic match information
            competition = match[0] if len(match) > 0 else None
            datetime_str = match[1] if len(match) > 1 else None
            season = match[2] if len(match) > 2 else None
            teams = match[3] if len(match) > 3 and isinstance(match[3], dict) else {}
            
            # Extract team information
            home_team = teams.get('home', None)
            away_team = teams.get('away', None)
            
            # Process events (home and away)
            events_lists = match[4:] if len(match) > 4 else []
            
            # If no events, create one row with match info only
            if not any(events_lists):
                row = {
                    'match_id': match_idx,
                    'competition': competition,
                    'datetime': datetime_str,
                    'season': season,
                    'home_team': home_team,
                    'away_team': away_team,
                    'event_type': None,
                    'time': None,
                    'player': None,
                    'team_side': None,
                    'score': None,
                    'assist': None,
                    'player_in': None,
                    'player_out': None,
                    'reason': None,
                    'event_source': None
                }
                all_rows.append(row)
            else:
                # Process each events list (typically home and away)
                for list_idx, events_list in enumerate(events_lists):
                    if not isinstance(events_list, list):
                        continue
                        
                    team_side = 'home' if list_idx == 0 else 'away'
                    
                    # If events list is empty, create one row for this team
                    if not events_list:
                        row = {
                            'match_id': match_idx,
                            'competition': competition,
                            'datetime': datetime_str,
                            'season': season,
                            'home_team': home_team,
                            'away_team': away_team,
                            'event_type': None,
                            'time': None,
                            'player': None,
                            'team_side': team_side,
                            'score': None,
                            'assist': None,
                            'player_in': None,
                            'player_out': None,
                            'reason': None,
                            'event_source': f'list_{list_idx}'
                        }
                        all_rows.append(row)
                        continue
                    
                    # Process each event in the list
                    for event in events_list:
                        if not isinstance(event, dict):
                            continue
                            
                        # Create base row with match information
                        row = {
                            'match_id': match_idx,
                            'competition': competition,
                            'datetime': datetime_str,
                            'season': season,
                            'home_team': home_team,
                            'away_team': away_team,
                            'team_side': team_side,
                            'event_source': f'list_{list_idx}'
                        }
                        
                        # Extract event information with flexible key handling
                        row['event_type'] = event.get('type', None)
                        row['time'] = event.get('time', None)
                        row['player'] = event.get('player', None)
                        row['score'] = event.get('score', None)
                        row['assist'] = event.get('assist', None)
                        row['player_in'] = event.get('player_in', None)
                        row['player_out'] = event.get('player_out', None)
                        row['reason'] = event.get('reason', None)
                        
                        # Override team_side if specified in event
                        if 'team' in event:
                            row['team_side'] = event['team']
                        
                        # Add any additional keys that might exist
                        for key, value in event.items():
                            if key not in ['type', 'time', 'player', 'score', 'assist', 
                                         'player_in', 'player_out', 'reason', 'team']:
                                row[f'extra_{key}'] = value
                        
                        all_rows.append(row)
                        
        except Exception as e:
            print(f"Error processing match {match_idx}: {e}")
            continue
    
    # Convert to DataFrame
    df = pd.DataFrame(all_rows)
    
    # Reorder columns for better readability
    base_columns = ['match_id', 'competition', 'datetime', 'season', 'home_team', 'away_team', 
                   'team_side', 'event_source', 'event_type', 'time', 'player', 'score', 
                   'assist', 'player_in', 'player_out', 'reason']
    
    # Add any extra columns that were found
    extra_columns = [col for col in df.columns if col.startswith('extra_')]
    final_columns = base_columns + extra_columns
    
    # Only include columns that exist in the DataFrame
    final_columns = [col for col in final_columns if col in df.columns]
    
    return df[final_columns]


def analyze_match_data(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Provide a summary analysis of the extracted match data.
    
    Args:
        df: DataFrame returned by extract_matches_to_dataframe
    
    Returns:
        Dictionary with analysis summary
    """
    analysis = {
        'total_matches': df['match_id'].nunique(),
        'total_events': len(df[df['event_type'].notna()]),
        'competitions': df['competition'].unique().tolist(),
        'seasons': df['season'].unique().tolist(),
        'event_types': df['event_type'].value_counts().to_dict(),
        'teams': sorted(set(df['home_team'].dropna().tolist() + df['away_team'].dropna().tolist())),
        'date_range': {
            'earliest': df['datetime'].min(),
            'latest': df['datetime'].max()
        }
    }
    
    return analysis


def process_match_dataframe(df):
    """
    Process an existing DataFrame to:
    1. Split 'competition' column into 'competition' and 'matchweek'
    2. Add 'half_event' column based on 'time' column
    
    Args:
        df: pandas DataFrame with 'competition' and 'time' columns
    
    Returns:
        pandas DataFrame with processed columns
    """
    # Make a copy to avoid modifying the original
    df_processed = df.copy()
    
    # Split competition column
    df_processed = split_competition_column(df_processed)
    
    # Add half_event column
    df_processed = add_half_event_column(df_processed)
    
    return df_processed

def split_competition_column(df):
    """
    Split the 'competition' column into 'competition' and 'matchweek' columns.
    """
    df = df.copy()
    
    # Initialize new columns
    df['matchweek'] = None
    
    # Function to split individual competition strings
    def split_single_competition(comp_str):
        if pd.isna(comp_str) or not isinstance(comp_str, str):
            return comp_str, None
        
        # Common patterns for matchweek information
        patterns = [
            r'^(.+?)\s*-\s*(JORNADA\s+\d+)$',   # "PREMIER LEAGUE - JORNADA 38"
            r'^(.+?)\s*-\s*(MATCHDAY\s+\d+)$',  # "PREMIER LEAGUE - MATCHDAY 38"
            r'^(.+?)\s*-\s*(GAMEWEEK\s+\d+)$',  # "PREMIER LEAGUE - GAMEWEEK 38"
            r'^(.+?)\s*-\s*(WEEK\s+\d+)$',      # "PREMIER LEAGUE - WEEK 38"
            r'^(.+?)\s*-\s*(MD\s*\d+)$',        # "PREMIER LEAGUE - MD38"
            r'^(.+?)\s*-\s*(GW\s*\d+)$',        # "PREMIER LEAGUE - GW38"
            r'^(.+?)\s*-\s*(ROUND\s+\d+)$',     # "PREMIER LEAGUE - ROUND 38"
            r'^(.+?)\s*-\s*(\d+)$',             # "PREMIER LEAGUE - 38"
        ]
        
        for pattern in patterns:
            match = re.match(pattern, comp_str.strip(), re.IGNORECASE)
            if match:
                competition_name = match.group(1).strip()
                matchweek = match.group(2).strip()
                return competition_name, matchweek
        
        # If no pattern matches, return the full string as competition name
        return comp_str.strip(), None
    
    # Apply the splitting function
    split_results = df['competition'].apply(split_single_competition)
    
    # Update the columns
    df['competition'] = [result[0] for result in split_results]
    df['matchweek'] = [result[1] for result in split_results]
    
    return df

def add_half_event_column(df):
    """
    Add 'half_event' column based on the 'time' column.
    """
    df = df.copy()
    
    def determine_half(time_str):
        """
        Determine if an event occurred in the first or second half.
        """
        if pd.isna(time_str) or not isinstance(time_str, str):
            return None
        
        # Extract the base minute from formats like "74'", "45+2'", "90+1'"
        match = re.match(r'(\d+)', str(time_str).strip())
        if not match:
            return None
        
        try:
            minute = int(match.group(1))
            
            # Football halves: 1-45 minutes = First Half, 46+ minutes = Second Half
            if 1 <= minute <= 45:
                return 'First Half'
            elif minute >= 46:
                return 'Second Half'
            else:
                return None
        except ValueError:
            return None
    
    # Add the half_event column
    df['half_event'] = df['time'].apply(determine_half)
    
    return df

def reorder_columns(df):
    """
    Reorder columns for better readability, putting new columns in logical positions.
    """
    # Define preferred column order
    preferred_order = [
        'match_id', 'competition', 'matchweek', 'datetime', 'season', 
        'home_team', 'away_team', 'team_side', 'event_source', 
        'event_type', 'time', 'half_event', 'player', 'score', 
        'assist', 'player_in', 'player_out', 'reason'
    ]
    
    # Get existing columns
    existing_cols = df.columns.tolist()
    
    # Start with preferred columns that exist
    final_order = [col for col in preferred_order if col in existing_cols]
    
    # Add any remaining columns that weren't in the preferred list
    remaining_cols = [col for col in existing_cols if col not in final_order]
    final_order.extend(remaining_cols)
    
    return df[final_order]

In [10]:
for i in range(1,12):
    print(i)

1
2
3
4
5
6
7
8
9
10
11


In [12]:
for i in range(1,12):
    print(i)
    with open(f'data_league{i}.json', 'r', encoding='utf-8') as f:
        read_data = json.load(f)

    df = extract_matches_to_dataframe(read_data)

    df2 = process_match_dataframe(df)

    goals_stats_teams = []

    unique_seasons = list(df2['season'].unique())

    for individual_season in unique_seasons:
        unique_teams = list(df2[df2['season'] == individual_season]['home_team'].unique())
        for individual_team in unique_teams:

            temporal_dict = {}

            home_goals_per_season = len(df2[(df2['home_team'] == individual_team) & (df2['event_type'] == 'goal') & (df2['team_side'] == 'home') & (df2['season'] == individual_season)])
            away_goals_per_season = len(df2[(df2['away_team'] == individual_team) & (df2['event_type'] == 'goal') & (df2['team_side'] == 'away') & (df2['season'] == individual_season)])

            average_goals_per_season = (home_goals_per_season + away_goals_per_season)/((len(unique_teams)-1)*2)

            # first half home season goals scored
            first_half_home_season_goals_scored = len(df2[(df2['home_team'] == individual_team) & (df2['event_type'] == 'goal') & (df2['team_side'] == 'home') & (df2['season'] == individual_season) & (df2['half_event'] == 'First Half')])
            # second half home season goals scored
            second_half_home_season_goals_scored = len(df2[(df2['home_team'] == individual_team) & (df2['event_type'] == 'goal') & (df2['team_side'] == 'home') & (df2['season'] == individual_season) & (df2['half_event'] == 'Second Half')])
            # first half away season goals scored
            first_half_away_season_goals_scored = len(df2[(df2['away_team'] == individual_team) & (df2['event_type'] == 'goal') & (df2['team_side'] == 'away') & (df2['season'] == individual_season) & (df2['half_event'] == 'First Half')])
            # first half away season goals scored
            second_half_away_season_goals_scored = len(df2[(df2['away_team'] == individual_team) & (df2['event_type'] == 'goal') & (df2['team_side'] == 'away') & (df2['season'] == individual_season) & (df2['half_event'] == 'Second Half')])
            # first half season goals average scored
            average_first_half_goals_scored = (first_half_home_season_goals_scored + first_half_away_season_goals_scored)/((len(unique_teams)-1)*2)
            # second half season goals average scored
            average_second_half_goals_scored = (second_half_away_season_goals_scored + second_half_home_season_goals_scored)/((len(unique_teams)-1)*2)

            average_home_game_first_half_goals_scored = first_half_home_season_goals_scored/(len(unique_teams)-1)

            average_home_game_second_half_goals_scored = second_half_home_season_goals_scored/(len(unique_teams)-1)

            average_away_game_first_half_goals_scored = first_half_away_season_goals_scored/(len(unique_teams)-1)

            average_away_game_second_half_goals_scored = second_half_away_season_goals_scored/(len(unique_teams)-1)


            # print(f'season: {individual_season} , team: {individual_team}: home goals: {home_goals_per_season}, away goals: {away_goals_per_season}, average goals: {average_goals_per_season}')
            # print('==============================')
            # print(f'first half all season goals scored: {first_half_home_season_goals_scored + first_half_away_season_goals_scored} second half all season goals scored: {second_half_away_season_goals_scored + second_half_home_season_goals_scored}')
            # print('==============================')
            # print(f'average goals first half: {average_first_half_goals_scored} average goals second half: {average_second_half_goals_scored}')
            # print('==============================')
            # print(f'home game first half goals scored: {first_half_home_season_goals_scored} home game second half goals scored {second_half_home_season_goals_scored}')
            # print('==============================')
            # print(f'average home game first half goals: {average_home_game_first_half_goals_scored} average home game second half goals: {average_home_game_second_half_goals_scored}')
            # print('==============================')
            # print(f'away game first half goals scored:{first_half_away_season_goals_scored}  away game second half goals scored:{second_half_away_season_goals_scored}')
            # print('==============================')
            # print(f'average away game first half goals: {average_away_game_first_half_goals_scored} average away game second half goals: {average_away_game_second_half_goals_scored}')

            temporal_dict.update({'season': individual_season, 'team': individual_team, 'home_goals_season': home_goals_per_season, 'away_goals_season':away_goals_per_season,
                                'average_goals_season':average_goals_per_season, 'first_half_season_goals_scored': first_half_home_season_goals_scored + first_half_away_season_goals_scored,
                                'second_half_season_goals_scored':second_half_away_season_goals_scored + second_half_home_season_goals_scored, 'average_season_first_half_goals_scored':average_first_half_goals_scored,
                                'average_season_second_half_goals_scored':average_second_half_goals_scored,'first_half_home_season_goals_scored':first_half_home_season_goals_scored,
                                'second_half_home_season_goals_scored':second_half_home_season_goals_scored, 'average_home_game_first_half_goals_scored':average_home_game_first_half_goals_scored,
                                'average_home_game_second_half_goals_scored':average_home_game_second_half_goals_scored,'first_half_away_season_goals_scored':first_half_away_season_goals_scored,
                                'second_half_away_season_goals_scored':second_half_away_season_goals_scored,'average_away_game_first_half_goals_scored':average_away_game_first_half_goals_scored,
                                'average_away_game_second_half_goals_scored':average_away_game_second_half_goals_scored})
            print(temporal_dict)
            goals_stats_teams.append(temporal_dict)
    with open(f'goal_stats_league{0}.json', 'w', encoding='utf-8') as f:
            json.dump(temporal_dict, f, ensure_ascii=False, indent=4)    


    i += 1

1
{'season': '2024/2025', 'team': 'Nottingham Forest', 'home_goals_season': 26, 'away_goals_season': 32, 'average_goals_season': 1.5263157894736843, 'first_half_season_goals_scored': 29, 'second_half_season_goals_scored': 29, 'average_season_first_half_goals_scored': 0.7631578947368421, 'average_season_second_half_goals_scored': 0.7631578947368421, 'first_half_home_season_goals_scored': 14, 'second_half_home_season_goals_scored': 12, 'average_home_game_first_half_goals_scored': 0.7368421052631579, 'average_home_game_second_half_goals_scored': 0.631578947368421, 'first_half_away_season_goals_scored': 15, 'second_half_away_season_goals_scored': 17, 'average_away_game_first_half_goals_scored': 0.7894736842105263, 'average_away_game_second_half_goals_scored': 0.8947368421052632}
{'season': '2024/2025', 'team': 'Southampton', 'home_goals_season': 13, 'away_goals_season': 13, 'average_goals_season': 0.6842105263157895, 'first_half_season_goals_scored': 12, 'second_half_season_goals_scored': 

KeyboardInterrupt: 

In [27]:
individual_season

'2024/2025'

In [ ]:
print(len(df2[(df2['home_team'] == 'Fulham') & (df2['event_type'] == 'goal') & (df2['team_side'] == 'home') & (df2['season'] == individual_season)]))
print(len(df2[(df2['away_team'] == 'Fulham') & (df2['event_type'] == 'goal') & (df2['team_side'] == 'away') & (df2['season'] == individual_season)]))

28
24


In [36]:
len(df2[(df2['home_team'] == 'Fulham') & (df2['event_type'] == 'goal') & (df2['team_side'] == 'away') & (df2['season'] == individual_season)])

28

In [39]:
len(df2[(df2['away_team'] == 'Fulham') & (df2['event_type'] == 'goal') & (df2['team_side'] == 'home') & (df2['season'] == individual_season)])

24

In [41]:
goals_stats_teams

[{'season': '2024/2025',
  'team': 'Nottingham Forest',
  'home_goals_season': 26,
  'away_goals_season': 32,
  'average_goals_season': 1.5263157894736843,
  'first_half_season_goals_scored': 29,
  'second_half_season_goals_scored': 29,
  'average_season_first_half_goals_scored': 0.7631578947368421,
  'average_season_second_half_goals_scored': 0.7631578947368421,
  'first_half_home_season_goals_scored': 14,
  'second_half_home_season_goals_scored': 12,
  'average_home_game_first_half_goals_scored': 0.7368421052631579,
  'average_home_game_second_half_goals_scored': 0.631578947368421,
  'first_half_away_season_goals_scored': 15,
  'second_half_away_season_goals_scored': 17,
  'average_away_game_first_half_goals_scored': 0.7894736842105263,
  'average_away_game_second_half_goals_scored': 0.8947368421052632},
 {'season': '2024/2025',
  'team': 'Southampton',
  'home_goals_season': 13,
  'away_goals_season': 13,
  'average_goals_season': 0.6842105263157895,
  'first_half_season_goals_score

In [ ]:
df['home_team'] 

,match_id,competition,datetime,season,home_team,away_team,team_side,event_source,event_type,time,player,score,assist,player_in,player_out,reason
0,0,PREMIER LEAGUE - JORNADA 38,25.05.2025 10:00,2024/2025,Nottingham Forest,Chelsea,home,list_0,yellow_card,32',Anderson E.,None,None,None,None,(Derribar a un rival)
1,0,PREMIER LEAGUE - JORNADA 38,25.05.2025 10:00,2024/2025,Nottingham Forest,Chelsea,home,list_0,substitution,57',Hudson-Odoi C.,None,None,Hudson-Odoi C.,Sangaré I.,None
2,0,PREMIER LEAGUE - JORNADA 38,25.05.2025 10:00,2024/2025,Nottingham Forest,Chelsea,home,list_0,substitution,68',Yates R.,None,None,Yates R.,Dominguez N.,None
3,0,PREMIER LEAGUE - JORNADA 38,25.05.2025 10:00,2024/2025,Nottingham Forest,Chelsea,home,list_0,yellow_card,79',Aina O.,None,None,None,None,(Falta)
4,0,PREMIER LEAGUE - JORNADA 38,25.05.2025 10:00,2024/2025,Nottingham Forest,Chelsea,home,list_0,substitution,83',Jota Silva,None,None,Jota Silva,Aina O.,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85564,7027,PREMIER LEAGUE - JORNADA 1,19.08.2006 06:45,2006/2007,Sheffield Utd,Liverpool,away,list_1,substitution,34',Agger D.,None,None,Agger D.,Carragher J.,None
85565,7027,PREMIER LEAGUE - JORNADA 1,19.08.2006 06:45,2006/2007,Sheffield Utd,Liverpool,away,list_1,yellow_card,64',Sissoko M.,None,None,None,None,None
85566,7027,PREMIER LEAGUE - JORNADA 1,19.08.2006 06:45,2006/2007,Sheffield Utd,Liverpool,away,list_1,goal,70',Fowler R.,1 - 1,None,None,None,None
85567,7027,PREMIER LEAGUE - JORNADA 1,19.08.2006 06:45,2006/2007,Sheffield Utd,Liverpool,away,list_1,yellow_card,82',Kromkamp J.,None,None,None,None,None


In [51]:
df2

,match_id,competition,datetime,season,home_team,away_team,team_side,event_source,event_type,time,player,score,assist,player_in,player_out,reason,matchweek,half_event
0,0,PREMIER LEAGUE,25.05.2025 10:00,2024/2025,Nottingham Forest,Chelsea,home,list_0,yellow_card,32',Anderson E.,None,None,None,None,(Derribar a un rival),JORNADA 38,First Half
1,0,PREMIER LEAGUE,25.05.2025 10:00,2024/2025,Nottingham Forest,Chelsea,home,list_0,substitution,57',Hudson-Odoi C.,None,None,Hudson-Odoi C.,Sangaré I.,None,JORNADA 38,Second Half
2,0,PREMIER LEAGUE,25.05.2025 10:00,2024/2025,Nottingham Forest,Chelsea,home,list_0,substitution,68',Yates R.,None,None,Yates R.,Dominguez N.,None,JORNADA 38,Second Half
3,0,PREMIER LEAGUE,25.05.2025 10:00,2024/2025,Nottingham Forest,Chelsea,home,list_0,yellow_card,79',Aina O.,None,None,None,None,(Falta),JORNADA 38,Second Half
4,0,PREMIER LEAGUE,25.05.2025 10:00,2024/2025,Nottingham Forest,Chelsea,home,list_0,substitution,83',Jota Silva,None,None,Jota Silva,Aina O.,None,JORNADA 38,Second Half
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85564,7027,PREMIER LEAGUE,19.08.2006 06:45,2006/2007,Sheffield Utd,Liverpool,away,list_1,substitution,34',Agger D.,None,None,Agger D.,Carragher J.,None,JORNADA 1,First Half
85565,7027,PREMIER LEAGUE,19.08.2006 06:45,2006/2007,Sheffield Utd,Liverpool,away,list_1,yellow_card,64',Sissoko M.,None,None,None,None,None,JORNADA 1,Second Half
85566,7027,PREMIER LEAGUE,19.08.2006 06:45,2006/2007,Sheffield Utd,Liverpool,away,list_1,goal,70',Fowler R.,1 - 1,None,None,None,None,JORNADA 1,Second Half
85567,7027,PREMIER LEAGUE,19.08.2006 06:45,2006/2007,Sheffield Utd,Liverpool,away,list_1,yellow_card,82',Kromkamp J.,None,None,None,None,None,JORNADA 1,Second Half


In [48]:
df2[(df2['home_team'] == 'Fulham') & (df2['event_type'] == 'goal') & (df2['season'] == individual_season)]

,match_id,competition,datetime,season,home_team,away_team,team_side,event_source,event_type,time,player,score,assist,player_in,player_out,reason,matchweek,half_event
320,20,PREMIER LEAGUE,10.05.2025 09:00,2024/2025,Fulham,Everton,home,list_0,goal,17',Jiménez R.,1 - 0,Smith Rowe E.,None,None,None,JORNADA 36,First Half
327,20,PREMIER LEAGUE,10.05.2025 09:00,2024/2025,Fulham,Everton,away,list_1,goal,45+3',Mykolenko V.,1 - 1,Doucouré A.,None,None,None,JORNADA 36,First Half
330,20,PREMIER LEAGUE,10.05.2025 09:00,2024/2025,Fulham,Everton,away,list_1,goal,70',Keane M.,1 - 2,McNeil D.,None,None,None,JORNADA 36,Second Half
331,20,PREMIER LEAGUE,10.05.2025 09:00,2024/2025,Fulham,Everton,away,list_1,goal,74',Beto,1 - 3,Alcaraz C.,None,None,None,JORNADA 36,Second Half
726,46,PREMIER LEAGUE,20.04.2025 08:00,2024/2025,Fulham,Chelsea,home,list_0,goal,20',Iwobi A.,1 - 0,Sessegnon R.,None,None,None,JORNADA 33,First Half
738,46,PREMIER LEAGUE,20.04.2025 08:00,2024/2025,Fulham,Chelsea,away,list_1,goal,83',George T.,1 - 1,None,None,None,None,JORNADA 33,Second Half
741,46,PREMIER LEAGUE,20.04.2025 08:00,2024/2025,Fulham,Chelsea,away,list_1,goal,90+3',Neto P.,1 - 2,Fernández E.,None,None,None,JORNADA 33,Second Half
1082,68,PREMIER LEAGUE,06.04.2025 08:00,2024/2025,Fulham,Liverpool,home,list_0,goal,23',Sessegnon R.,1 - 1,None,None,None,None,JORNADA 31,First Half
1083,68,PREMIER LEAGUE,06.04.2025 08:00,2024/2025,Fulham,Liverpool,home,list_0,goal,32',Iwobi A.,2 - 1,None,None,None,None,JORNADA 31,First Half
1084,68,PREMIER LEAGUE,06.04.2025 08:00,2024/2025,Fulham,Liverpool,home,list_0,goal,37',Rodrigo Muniz,3 - 1,Iwobi A.,None,None,None,JORNADA 31,First Half
